<a href="https://colab.research.google.com/github/Leopqs/Treinamento_de_IAs/blob/main/2bimestre/trabalho/algoritmoStacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# ALGORITMO STACKING COM SMOTETOMEK
# Dataset: Diabetes Health Indicators
# ============================================

# ==============================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTETomek


# ==============================
# 2. CARREGAMENTO DA BASE
# ==============================

# Deixe o arquivo CSV na mesma pasta deste código
df = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv")

print("Primeiras linhas da base:")
print(df.head())

print("\nInformações da base:")
print(df.info())

print("\nDistribuição original da variável alvo:")
print(df["Diabetes_binary"].value_counts())

print("\nDistribuição original em porcentagem:")
print(df["Diabetes_binary"].value_counts(normalize=True) * 100)


# ==============================
# 3. SEPARAÇÃO ENTRE ENTRADAS E ALVO
# ==============================

# X recebe as colunas usadas para prever
X = df.drop("Diabetes_binary", axis=1)

# y recebe a coluna que queremos prever
y = df["Diabetes_binary"]

print("\nQuantidade original de classes:")
print(Counter(y))


# ==============================
# 4. DIVISÃO ENTRE TREINO E TESTE
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nClasses no treino antes do balanceamento:")
print(Counter(y_train))

print("\nClasses no teste:")
print(Counter(y_test))


# ==============================
# 5. MODELOS BASE DO STACKING
# ==============================

modelos_base = [
    (
        "random_forest",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        )
    ),
    (
        "gradient_boosting",
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        )
    ),
    (
        "knn",
        KNeighborsClassifier(
            n_neighbors=7
        )
    )
]


# ==============================
# 6. META-MODELO DO STACKING
# ==============================

meta_modelo = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)


# ==============================
# 7. MODELO STACKING
# ==============================

stacking_model = StackingClassifier(
    estimators=modelos_base,
    final_estimator=meta_modelo,
    cv=3,
    stack_method="predict_proba",
    n_jobs=-1
)


# ==============================
# 8. PIPELINE FINAL
# ==============================

pipeline = Pipeline(steps=[
    # Padroniza os dados
    ("scaler", StandardScaler()),

    # Aplica SMOTE + Tomek Links
    # SMOTE cria exemplos sintéticos da classe minoritária
    # Tomek Links remove exemplos muito próximos entre classes diferentes
    ("smote_tomek", SMOTETomek(random_state=42)),

    # Modelo preditivo final
    ("stacking", stacking_model)
])


# ==============================
# 9. TREINAMENTO DO MODELO
# ==============================

print("\nTreinando o modelo...")

pipeline.fit(X_train, y_train)

print("Treinamento finalizado!")


# ==============================
# 10. PREDIÇÕES
# ==============================

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]


# ==============================
# 11. MÉTRICAS DE AVALIAÇÃO
# ==============================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("\n========== MÉTRICAS DO MODELO STACKING ==========")
print(f"Acurácia:  {accuracy:.4f}")
print(f"Precisão:  {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))


# ==============================
# 12. MATRIZ DE CONFUSÃO
# ==============================

matriz = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Sem diabetes", "Diabetes/Pré-diabetes"],
    yticklabels=["Sem diabetes", "Diabetes/Pré-diabetes"]
)

plt.title("Matriz de Confusão - Stacking com SMOTETomek")
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.show()


# ==============================
# 13. CURVA ROC
# ==============================

RocCurveDisplay.from_predictions(y_test, y_proba)

plt.title("Curva ROC - Stacking com SMOTETomek")
plt.show()

Primeiras linhas da base:
   Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1              0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2              0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3              0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4              0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      5.0      18.0      15